# Run experiment on IT monitor datasets

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import time
import os
import graphviz as gv
import math


print([np.__version__, pd.__version__])
np.set_printoptions(precision=3, suppress=True)


from src.data_preprocessing import preprocess_data
from src.plotting import plot_heatmap, plot_causal_graph
from src.causal_matrix_evaluation import evaluate_causal_matrices
from src.matrix_utils import read_matrices_from_csv, save_matrices, get_summary_matrix
from src.run_causal_discovery import run_varlingam, run_varlingam_bootstrap, run_pcmci, run_rcd_varlingam, run_rcd_pcmci, run_tcdf

In [2]:
def run_experiments(activity_types, n_datasets=2, methods=['varlingam', 'pcmci', 'rcd_varlingam', 'rcd_pcmci', 'tcdf']):
    """
    Run causal discovery experiments on IT monitor datasets.
    
    Args:
        activity_types: List of data types to process
        n_datasets: Number of datasets per type
        methods: List of methods to run
    """
    for activity_type in activity_types:
        print(f"\nRunning experiments for {activity_type}...")
        
        # Load ground truths
        ground_truths_path = f'data/real/IT_monitor/{activity_type}/ground_truth.csv'
        ground_truths = read_matrices_from_csv(ground_truths_path)
        
        if ground_truths is None:
            print(f"Skipping {activity_type} due to missing ground truths")
            continue
            
        ground_truth = ground_truths[0]  # Use first matrix as ground truth
        
        for method in methods:
            results = []
            
            for i in range(n_datasets):
                # Load and preprocess data
                data_path = f'data/real/IT_monitor/{activity_type}/dataset_{i}.csv'
                data = pd.read_csv(data_path)
                columns = data.columns.tolist()
                
                # Remove timestamp if present
                for time_col in ['Date', 'timestamp']:
                    if time_col in columns:
                        data = data.drop([time_col], axis=1)
                        columns.remove(time_col)
                
                data = data.values
                data = preprocess_data(data, columns)
                
                print(f"Running {method} on dataset {i} of {activity_type}")
                
                # Run causal discovery method
                start_time = time.time()
                
                if method == 'varlingam':
                    adjacency_matrices = run_varlingam(data)
                elif method == 'varlingam_bootstrap':
                    adjacency_matrices = run_varlingam_bootstrap(data)
                elif method == 'pcmci':
                    adjacency_matrices = run_pcmci(data)
                elif method == 'rcd_varlingam':
                    adjacency_matrices = run_rcd_varlingam(data)
                elif method == 'rcd_pcmci':
                    adjacency_matrices = run_rcd_pcmci(data)
                elif method == 'tcdf':
                    adjacency_matrices = run_tcdf(data)
                else:
                    raise ValueError(f"Unknown method: {method}")
                
                runtime = round(time.time() - start_time, 4)
                
                # Get summary matrix from method results
                summary_matrix = get_summary_matrix(adjacency_matrices)
                
                # Save results for first dataset
                if i == 0:
                    # Save summary matrix
                    summary_path = f'results/real/IT_monitor/{activity_type}/sum_adj_matrix_{method}.csv'
                    os.makedirs(os.path.dirname(summary_path), exist_ok=True)
                    save_matrices([summary_matrix], summary_path)
                    
                    # Save full adjacency matrices
                    matrices_path = f'results/real/IT_monitor/{activity_type}/adj_matrices_{method}.csv'
                    save_matrices(adjacency_matrices, matrices_path)
                
                # Evaluate summary matrix against ground truth
                evaluation = evaluate_causal_matrices([ground_truth], [summary_matrix])
                
                # Store results
                results.append({
                    'dataset': f'dataset_{i}',
                    'SHD': evaluation['shd'],
                    'F1': evaluation['f1'],
                    'F1_sign': evaluation['f1_sign'],
                    'runtime': runtime
                })
            
            # Calculate summary statistics
            averages = {
                metric: np.mean([r[metric] for r in results if isinstance(r[metric], (int, float))])
                for metric in ['SHD', 'F1', 'F1_sign', 'runtime']
            }
            
            # Add summary row
            results.append({
                'dataset': 'Overall Average',
                'SHD': f"{averages['SHD']:.2f}",
                'F1': f"{averages['F1']:.3f}",
                'F1_sign': f"{averages['F1_sign']:.3f}",
                'runtime': f"{averages['runtime']:.4f}"
            })
            
            # Save results
            df_results = pd.DataFrame(results)
            results_path = f'results/real/IT_monitor/{activity_type}/performance_{method}.csv'
            df_results.to_csv(results_path, index=False)

In [ ]:
# Define activity types to process
activity_types = [
    'Antivirus_Activity',
    'Middleware_oriented_message_Activity',
    'Web_Activity'
]

# Define methods to run
methods_to_run = ['varlingam', 'pcmci', 'rcd_varlingam', 'rcd_pcmci', 'tcdf']

# Run experiments
run_experiments(activity_types, n_datasets=2, methods=methods_to_run)